# Figure 1 — Failure Mode Distribution 
Redesigned version of fig1_failure_modes for the ICSSI extended abstract.

In [1]:
import json
from collections import Counter
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUT = Path('output')
FIG = OUT / 'figures'
FIG.mkdir(exist_ok=True)

In [2]:
# Load 10k annotation results
with open(OUT / 'integrity_all_10000.json') as f:
    all_10k = json.load(f)['entries']

N_10k = len(all_10k)
labels_10k = [e['claude_label'] for e in all_10k]

dist_10k = Counter(labels_10k)
n_valid_10k = dist_10k.pop('Valid')
n_rejected_10k = N_10k - n_valid_10k

fm_sorted = dist_10k.most_common()
fm_labels = [c[0] for c in fm_sorted]
fm_counts = [c[1] for c in fm_sorted]
total_fm = sum(fm_counts)

print(f'Valid: {n_valid_10k}, Rejected: {n_rejected_10k}, Total failure modes: {total_fm}')

Valid: 8795, Rejected: 1205, Total failure modes: 1205


In [22]:
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.28, 0.72],
    specs=[[{'type': 'pie'}, {'type': 'bar'}]],
    horizontal_spacing=0.22,   # increased to push bar y-axis labels away from the pie
)

# --- Left panel: pie chart ---
# rotation=310 positions "Rejected" on the right side of the pie, facing the bar chart
fig.add_trace(
    go.Pie(
        labels=['Valid', 'Rejected'],
        values=[n_valid_10k, n_rejected_10k],
        marker=dict(
            colors=['#7BB0DF', '#DB5829'],
            line=dict(color='white', width=2),
        ),
        textinfo='label+percent',
        texttemplate='<b>%{label}</b><br>%{value:,}<br>(%{percent})',
        textposition='inside',
        insidetextorientation='horizontal',
        textfont=dict(size=11, color='white'),
        hovertemplate='%{label}: %{value:,} (%{percent})<extra></extra>',
        hole=0,
        sort=False,
        direction='clockwise',
        rotation=310,  # Rejected slice faces right, toward the bar chart
        showlegend=False,
    ),
    row=1, col=1,
)

# --- Right panel: horizontal bar chart ---
# Reverse so largest is at top
bar_labels = fm_labels[::-1]
bar_counts = fm_counts[::-1]
bar_pcts = [c / total_fm * 100 for c in bar_counts]
bar_text = [f'{c}  ({p:.0f}%)' for c, p in zip(bar_counts, bar_pcts)]

fig.add_trace(
    go.Bar(
        y=bar_labels,
        x=bar_counts,
        orientation='h',
        marker=dict(color='#DB5829', line=dict(color='white', width=0.5)),  # matches Rejected slice
        text=bar_text,
        textposition='outside',
        textfont=dict(size=11, color='#333'),
        hovertemplate='%{y}: %{x}<extra></extra>',
        showlegend=False,
    ),
    row=1, col=2,
)

fig.update_layout(
    width=980,
    height=360,
    margin=dict(l=1, r=1, t=5, b=60),
    # title=dict(
    #     text=f'Distribution of abstract integrity failures across {N_10k:,} OpenAlex abstracts',
    #     font=dict(size=14),
    #     x=0.5,
    #     xanchor='center',
    # ),
    font=dict(size=11),
    plot_bgcolor='white',
    paper_bgcolor='white',
)

# Style the bar axis
fig.update_xaxes(
    title_text='Number of abstracts',
    row=1, col=2,
    showgrid=True,
    gridcolor='#EEEEEE',
    zeroline=False,
    range=[0, max(fm_counts) * 1.38],
)
fig.update_yaxes(
    row=1, col=2,
    tickfont=dict(size=11),
)

# "n = 10,000" below the pie
fig.add_annotation(
    text=f'<b>n = {N_10k:,}</b>',
    x=0.12, y=-0.06,
    xref='paper', yref='paper',
    showarrow=False,
    font=dict(size=11, color='#555'),
)

# "Breakdown of rejected" header above the bar chart, colored to match the Rejected slice
fig.add_annotation(
    text=f'<b>Breakdown of {total_fm:,} rejected</b>',
    x=0.70, y=1.07,
    xref='paper', yref='paper',
    showarrow=False,
    font=dict(size=11, color='#DB5829'),
)

fig.show()

In [23]:
# Export to PDF and PNG
fig.write_image(FIG / 'fig4_failure_modes_v2.pdf')
# fig.write_image(FIG / 'fig1_failure_modes_v2.png', scale=3)
# print('Saved to', FIG / 'fig1_failure_modes_v2.pdf')

# Figure 2 — Annotator Agreement (Plotly)

Rejection rates per annotator and pairwise Cohen's κ heatmap.

In [5]:
import json
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import cohen_kappa_score

OUT = Path('output')
FIG = OUT / 'figures'

with open(OUT / 'integrity_final_1000.json') as f:
    final_data = json.load(f)['entries']

a1  = np.array([e['a1']  for e in final_data])
a2 = np.array([e['a2'] for e in final_data])
claude = np.array([e['claude'] for e in final_data])
codex  = np.array([e['codex']  for e in final_data])

ann_names  = ['A1', 'A2', 'Claude', 'Codex']
ann_arrays = [a1, a2, claude, codex]
ann_types  = ['Human', 'Human', 'LLM', 'LLM']
ann_colors = ['#7BB0DF', '#B6DBFF', '#00A087', '#91D1C2']

# Rejection rates
rej_rates = [(1000 - a.sum()) / 10 for a in ann_arrays]

# Pairwise Cohen's κ matrix (4×4, diagonal = None)
kappa_matrix = [[None] * 4 for _ in range(4)]
kappa_text   = [[''   ] * 4 for _ in range(4)]
for i in range(4):
    for j in range(4):
        if i != j:
            k = cohen_kappa_score(ann_arrays[i], ann_arrays[j])
            kappa_matrix[i][j] = k
            kappa_text[i][j]   = f'{k:.2f}'

In [6]:
fig2 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.38, 0.62],
    horizontal_spacing=0.14,
    subplot_titles=['(a) Individual rejection rates', "(b) Pairwise Cohen's κ"],
)

# --- Panel A: rejection rate bars, one color per annotator ---
for i, (name, rate, color, atype) in enumerate(zip(ann_names, rej_rates, ann_colors, ann_types)):
    fig2.add_trace(
        go.Bar(
            x=[name],
            y=[rate],
            marker_color=color,
            marker_line=dict(color='white', width=1.5),
            text=[f'<b>{rate:.1f}%</b>'],
            textposition='outside',
            textfont=dict(size=12),
            hovertemplate=f'{name} ({atype}): {rate:.1f}%<extra></extra>',
            # showlegend=(i in [0, 2]),   # one legend entry per type
            showlegend=False,  # we'll add custom legend entries later
            legendgroup=atype,
            name=atype,
        ),
        row=1, col=1,
    )

# Legend patches for Human / LLM
fig2.add_trace(go.Bar(x=[None], y=[None], marker_color='#7BB0DF', name='Human',
                      legendgroup='Human', showlegend=False), row=1, col=1)
fig2.add_trace(go.Bar(x=[None], y=[None], marker_color='#DB5829', name='LLM',
                      legendgroup='LLM', showlegend=False), row=1, col=1)

fig2.update_xaxes(row=1, col=1, tickfont=dict(size=12))
fig2.update_yaxes(row=1, col=1,
                  title_text='Rejection rate (%)',
                  range=[0, 28],
                  showgrid=True, gridcolor='#EEEEEE', zeroline=False)

# --- Panel B: κ heatmap ---
# Mask diagonal with a neutral gray layer on top
kappa_vals = [[v if v is not None else 0.0 for v in row] for row in kappa_matrix]

fig2.add_trace(
    go.Heatmap(
        z=kappa_vals,
        x=ann_names,
        y=ann_names,
        text=kappa_text,
        texttemplate='<b>%{text}</b>',
        textfont=dict(size=14),
        colorscale=[
                [0.0,  '#E64B35'],
                [0.25, '#F39B7F'],
                [0.5,  '#F5F0E8'],
                [0.75, '#91D1C2'],
                [1.0,  '#00A087'],
            ],
        zmin=0.2, zmax=1.0,
        colorbar=dict(
            title=dict(text="Cohen's κ", side='right'),
            thickness=14,
            len=0.75,
        ),
        hovertemplate='%{y} vs %{x}: κ = %{text}<extra></extra>',
        showscale=True,
    ),
    row=1, col=2,
)

# Gray out the diagonal with invisible scatter markers carrying gray square shapes
# for i, name in enumerate(ann_names):
#     fig2.add_trace(
#         go.Scatter(
#             x=[name], y=[name],
#             mode='markers',
#             marker=dict(symbol='square', size=38, color='#E0E0E0', line=dict(width=0)),
#             hoverinfo='skip',
#             showlegend=False,
#         ),
#         row=1, col=2,
#     )

fig2.update_xaxes(row=1, col=2, side='bottom', tickfont=dict(size=12))
fig2.update_yaxes(row=1, col=2, autorange='reversed', tickfont=dict(size=12))

fig2.update_layout(
    width=900,
    height=380,
    margin=dict(l=20, r=20, t=50, b=50),
    # title=dict(
    #     text='Inter-annotator agreement across four annotators',
    #     font=dict(size=14),
    #     x=0.5, xanchor='center',
    # ),
    font=dict(size=11),
    plot_bgcolor='white',
    paper_bgcolor='white',
    barmode='group',
    legend=dict(
        orientation='v',
        x=0.32, y=0.98,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#CCC', borderwidth=1,
    ),
    # Subplot title styling
    annotations=[
        dict(text='(a) Individual rejection rates', x=0.17, y=1.06,
             xref='paper', yref='paper', showarrow=False, font=dict(size=12)),
        dict(text="(b) Pairwise Cohen's κ", x=0.72, y=1.06,
             xref='paper', yref='paper', showarrow=False, font=dict(size=12)),
    ],
)

fig2.show()

In [7]:
fig2.write_image(FIG / 'fig1_agreement_plotly.pdf')
# fig2.write_image(FIG / 'fig2_agreement_plotly.png', scale=3)
# print('Saved fig2')

Saved fig2


# Figure 3 — LLM vs. Human Ground Truth: Confusion Matrix (Plotly)

In [8]:
import json
import numpy as np
from pathlib import Path
import plotly.graph_objects as go

OUT = Path('output')
FIG = OUT / 'figures'

with open(OUT / 'integrity_final_1000.json') as f:
    final_data = json.load(f)['entries']
with open(OUT / 'integrity_all_10000.json') as f:
    all_10k = json.load(f)['entries']

final_map = {e['entry_id']: e['final_verdict'] for e in final_data}
gt, cl = [], []
for e in all_10k[:1000]:
    eid = e['entry_id']
    if eid in final_map:
        gt.append('Valid' if final_map[eid] else 'Rejected')
        cl.append('Valid' if e['claude_label'] == 'Valid' else 'Rejected')

# Build 2×2 matrix: rows = human GT, cols = Claude; order: Valid first, Rejected second
labels = ['Valid', 'Rejected']
cm = np.zeros((2, 2), dtype=int)
label_idx = {l: i for i, l in enumerate(labels)}
for g, c in zip(gt, cl):
    cm[label_idx[g]][label_idx[c]] += 1

total = cm.sum()
cell_text = [[f'<b>{cm[i][j]}</b><br>({cm[i][j]/total*100:.1f}%)'
              for j in range(2)] for i in range(2)]

cell_colors = [
    ['#7BB0DF', '#C5DCF2'],   # Human Valid row:   TP=dark blue,  FP=light blue
    ['#F5C0A0', '#DB5829'],   # Human Rejected row: FN=light orange, TN=dark orange
]
text_colors = [
    ['white', '#333'],
    ['#333', 'white'],
]

fig3 = go.Figure()

for i, gt_label in enumerate(labels):
    for j, cl_label in enumerate(labels):
        fig3.add_shape(
            type='rect',
            x0=j - 0.5, x1=j + 0.5,
            y0=i - 0.5, y1=i + 0.5,
            fillcolor=cell_colors[i][j],
            line=dict(color='white', width=3),
        )
        fig3.add_annotation(
            x=j, y=i,
            text=cell_text[i][j],
            showarrow=False,
            font=dict(size=18, color=text_colors[i][j], family='Arial'),
        )

fig3.update_xaxes(
    tickvals=[0, 1],
    ticktext=labels,
    tickfont=dict(size=13),
    title=dict(text='Claude Opus 4.6 (calibrated prompt)', font=dict(size=13)),
    range=[-0.5, 1.5],
    showgrid=False, zeroline=False,
    side='bottom',
)
fig3.update_yaxes(
    tickvals=[0, 1],
    ticktext=labels,
    tickfont=dict(size=13),
    title=dict(text='Human consensus (ground truth)', font=dict(size=13)),
    range=[1.5, -0.5],  # Valid (0) on top, Rejected (1) on bottom
    showgrid=False, zeroline=False,
)

n_correct = cm[0][0] + cm[1][1]
fig3.add_annotation(
    x=0.5, y=-0.22,
    xref='paper', yref='paper',
    text=f'<b>Overall agreement: {n_correct}/{total} ({n_correct/total*100:.1f}%)</b>',
    showarrow=False,
    font=dict(size=12, color='#333'),
)

fig3.update_layout(
    width=460,
    height=440,
    margin=dict(l=20, r=20, t=60, b=70),
    font=dict(family='Arial, sans-serif', size=11),
    plot_bgcolor='white',
    paper_bgcolor='white',
)

fig3.show()

In [9]:
fig3.write_image(FIG / 'fig2_confusion_matrix_plotly.pdf')
# fig3.write_image(FIG / 'fig3_confusion_matrix_plotly.png', scale=3)
print('Saved fig3')

Saved fig3


# Figures 6, 7, 8 — Temporal Analysis (Plotly)

Shared data setup and three figures converted from matplotlib with NPG colour scheme.

In [10]:
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

# Load 1k final data (needed for fig6 human ground truth)
with open(OUT / 'integrity_final_1000.json') as f:
    final_data_68 = json.load(f)['entries']

# Build DataFrames
df_10k = pd.DataFrame(all_10k)
df_1k  = pd.DataFrame(final_data_68)

# Join publication dates
dates = pd.read_csv(OUT / 'integrity_10000_publication_dates.csv')
m10k = df_10k.merge(dates[['entry_id', 'year']], on='entry_id', how='left')
m1k  = df_1k.merge(dates[['entry_id', 'year']], on='entry_id', how='left')

# Time periods
PERIOD_ORDER = ['Pre-2000', '2000–2009', '2010–2019', '2020–2025']

def assign_period(year):
    if pd.isna(year): return None
    if year < 2000: return 'Pre-2000'
    if year < 2010: return '2000–2009'
    if year < 2020: return '2010–2019'
    return '2020–2025'

m10k['period'] = m10k['year'].apply(assign_period)
m1k['period']  = m1k['year'].apply(assign_period)

periods_10k = m10k.groupby('period').apply(
    lambda g: pd.Series({
        'n': len(g),
        'n_rejected': (g['claude_label'] != 'Valid').sum(),
        'rate': (g['claude_label'] != 'Valid').mean() * 100,
    })
).reindex(PERIOD_ORDER)

periods_1k = m1k.groupby('period').apply(
    lambda g: pd.Series({
        'n': len(g),
        'n_rejected': (~g['final_verdict'].astype(bool)).sum(),
        'rate': (~g['final_verdict'].astype(bool)).mean() * 100,
    })
).reindex(PERIOD_ORDER)

print('Periods loaded.')
print(periods_10k[['n', 'n_rejected', 'rate']].to_string(float_format='%.1f'))


Periods loaded.
               n  n_rejected  rate
period                            
Pre-2000  2142.0       440.0  20.5
2000–2009 1923.0       226.0  11.8
2010–2019 3581.0       385.0  10.8
2020–2025 2354.0       154.0   6.5


/var/folders/xy/r3gq5vtd7bx6qb966l0fhh9h0000gn/T/ipykernel_83532/72256285.py:30: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/xy/r3gq5vtd7bx6qb966l0fhh9h0000gn/T/ipykernel_83532/72256285.py:38: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



## Figure 6 — Rejection Rate by Time Period

In [11]:
# Figure 6 — Rejection rate by time period

rates_1k  = periods_1k['rate'].values
n_1k      = periods_1k['n'].values.astype(int)
nrej_1k   = periods_1k['n_rejected'].values.astype(int)
ci_lo_1k, ci_hi_1k = proportion_confint(nrej_1k, n_1k, alpha=0.05, method='wilson')
err_hi_1k = ci_hi_1k * 100 - rates_1k
err_lo_1k = rates_1k - ci_lo_1k * 100

rates_10k  = periods_10k['rate'].values
n_10k      = periods_10k['n'].values.astype(int)
nrej_10k   = periods_10k['n_rejected'].values.astype(int)
ci_lo_10k, ci_hi_10k = proportion_confint(nrej_10k, n_10k, alpha=0.05, method='wilson')
err_hi_10k = ci_hi_10k * 100 - rates_10k
err_lo_10k = rates_10k - ci_lo_10k * 100

fig6 = go.Figure()

fig6.add_trace(go.Bar(
    name='Human ground truth (1k)',
    x=PERIOD_ORDER,
    y=rates_1k,
    error_y=dict(type='data', array=err_hi_1k, arrayminus=err_lo_1k, thickness=1.5, width=5),
    marker=dict(color='#B6DBFF', line=dict(color='white', width=1.5)),
    hovertemplate='%{x} — Human: %{y:.1f}% (n=%{customdata})<extra></extra>',
    customdata=n_1k,
))

fig6.add_trace(go.Bar(
    name='Claude annotation (10k)',
    x=PERIOD_ORDER,
    y=rates_10k,
    error_y=dict(type='data', array=err_hi_10k, arrayminus=err_lo_10k, thickness=1.5, width=5),
    marker=dict(color='#00A087', line=dict(color='white', width=1.5)),
    hovertemplate='%{x} — Claude: %{y:.1f}% (n=%{customdata})<extra></extra>',
    customdata=n_10k,
))

fig6.update_layout(
    width=720,
    height=400,
    barmode='group',
    margin=dict(l=20, r=20, t=20, b=50),
    font=dict(size=11),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(orientation='v', x=0.72, y=0.98,
                bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCC', borderwidth=1),
    yaxis=dict(
        title='Rejection rate (%)',
        range=[0, 28],
        showgrid=True, gridcolor='#EEEEEE', zeroline=False,
    ),
    xaxis=dict(tickfont=dict(size=12)),
)

# Add rate% + n= annotations at the midpoint of each bar.
# In a grouped bar with 2 traces the bars are offset by ±barwidth/2.
# Plotly default bargap=0.2, bargroupgap=0.0 → effective bar width ≈ 0.4 category units.
# The two bars are centered at x±0.2 in category-index space (0,1,2,3).
offset = 0.2   # adjust if bars look misaligned
for i, (period, r1, n1, r2, n2) in enumerate(
        zip(PERIOD_ORDER, rates_1k, n_1k, rates_10k, n_10k)):
    # Human bar (left)
    fig6.add_annotation(
        x=i - offset, y=r1 / 2,
        text=f'<b>{r1:.1f}%</b><br><span style="font-size:9px">n={n1:,}</span>',
        showarrow=False,
        font=dict(size=10, color='dark gray'),
        xref='x', yref='y',
    )
    # Claude bar (right)
    fig6.add_annotation(
        x=i + offset, y=r2 / 2,
        text=f'<b>{r2:.1f}%</b><br><span style="font-size:9px">n={n2:,}</span>',
        showarrow=False,
        font=dict(size=10, color='white'),
        xref='x', yref='y',
    )

fig6.show()


In [12]:
fig6.write_image(FIG / 'fig3_rejection_by_period.pdf')
# fig6.write_image(FIG / 'fig6_rejection_by_period.png', scale=3)
print('Saved fig6')


Saved fig6


## Figure 7 — Failure Mode Composition by Period

In [13]:
# Figure 7 — Failure mode composition by time period

FM_ORDER = [
    'Insufficient abstract content',
    'Bibliographic / repository metadata',
    'Wrong document section',
    'Web-scrape artefacts',
    'Truncated abstract',
    'No abstract / placeholder',
    'Wrong scholarly genre',
]
FM_SHORT = [
    'Insufficient content',
    'Bibliographic metadata',
    'Wrong section',
    'Web-scrape artefacts',
    'Truncated',
    'No abstract / placeholder',
    'Wrong genre',
]

# NPG 7-color palette (one per failure mode)
FM_COLORS = ['#B6DBFF', '#7BB0DF', '#91D1C2', '#00A087', '#F5F0E8', '#F39B7F', '#E64B35']

rejected_10k = m10k[m10k['claude_label'] != 'Valid']
comp = (
    rejected_10k.groupby(['period', 'claude_label']).size()
    .unstack(fill_value=0)
    .reindex(index=PERIOD_ORDER, columns=FM_ORDER, fill_value=0)
)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100
period_ns = comp.sum(axis=1)  # total rejected per period

fig7 = go.Figure()

for fm, short, color in zip(FM_ORDER, FM_SHORT, FM_COLORS):
    vals = comp_pct[fm].values
    text_labels = [f'{v:.0f}%' if v >= 8 else '' for v in vals]
    fig7.add_trace(go.Bar(
        name=short,
        x=[f'{p}<br>(n={int(period_ns[p])})' for p in PERIOD_ORDER],
        y=vals,
        marker=dict(color=color, line=dict(color='white', width=1)),
        text=text_labels,
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(size=11, color='dark gray', weight='bold'),
        hovertemplate='%{x}<br>' + short + ': %{y:.1f}%<extra></extra>',
    ))

fig7.update_layout(
    width=820,
    height=460,
    barmode='stack',
    margin=dict(l=20, r=200, t=20, b=60),
    font=dict(size=11),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        title=dict(text='Failure mode', font=dict(size=11)),
        orientation='v',
        x=1.01, y=1,
        xanchor='left',
        bgcolor='rgba(255,255,255,0.85)',
        # bordercolor='#CCC', borderwidth=1,
        font=dict(size=10),
    ),
    yaxis=dict(
        title='Share of rejected abstracts (%)',
        range=[0, 103],
        showgrid=False, gridcolor='#EEEEEE', zeroline=False,
    ),
    xaxis=dict(tickfont=dict(size=11)),
)

fig7.show()


In [14]:
fig7.write_image(FIG / 'fig5_failure_mode_by_period.pdf')
# fig7.write_image(FIG / 'fig7_failure_mode_by_period.png', scale=3)
print('Saved fig7')


Saved fig7


## Figure 8 — Year-Level Sample Size and Rejection Rate

In [15]:
# Figure 8 — Year-level sample size and rejection rate

yearly = (
    m10k.groupby('year')
    .agg(n=('entry_id', 'size'),
         n_rejected=('claude_label', lambda s: (s != 'Valid').sum()))
    .reset_index()
)
yearly['rate'] = yearly['n_rejected'] / yearly['n'] * 100
yearly = yearly.sort_values('year')
yearly['rej_rolling'] = (
    yearly['n_rejected'].rolling(5, min_periods=1, center=True).sum()
    / yearly['n'].rolling(5, min_periods=1, center=True).sum() * 100
)
yearly = yearly[yearly['n'] > 0].reset_index(drop=True)

overall_rate = yearly['n_rejected'].sum() / yearly['n'].sum() * 100

fig8 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.38, 0.62],
    vertical_spacing=0.04,
)

# Top panel: sample counts
fig8.add_trace(
    go.Bar(
        x=yearly['year'],
        y=yearly['n'],
        marker=dict(color='#B0BEC5', line=dict(width=0)),
        name='10k sample',
        hovertemplate='%{x}: %{y} abstracts<extra></extra>',
    ),
    row=1, col=1,
)

# Bottom panel: per-year scatter (size ∝ n)
fig8.add_trace(
    go.Scatter(
        x=yearly['year'],
        y=yearly['rate'],
        mode='markers',
        marker=dict(
            size=(yearly['n'] * 0.8).clip(3, 40),
            color='#F39B7F',
            opacity=0.55,
            line=dict(width=0),
        ),
        name='Per-year rate (size ∝ n)',
        hovertemplate='%{x}: %{y:.1f}%<extra></extra>',
    ),
    row=2, col=1,
)

# 5-year rolling average
fig8.add_trace(
    go.Scatter(
        x=yearly['year'],
        y=yearly['rej_rolling'],
        mode='lines',
        line=dict(color='#E64B35', width=2.5),
        name='5-year rolling average',
        hovertemplate='%{x}: %{y:.1f}% (rolling)<extra></extra>',
    ),
    row=2, col=1,
)

# Overall rate reference line
fig8.add_hline(
    y=overall_rate,
    line=dict(color='#888', width=1, dash='dot'),
    row=2, col=1,
    annotation_text=f'Overall: {overall_rate:.1f}%',
    annotation_position='bottom left',
    annotation_font=dict(size=10, color='#888'),
)

fig8.update_layout(
    width=900,
    height=520,
    margin=dict(l=20, r=20, t=20, b=50),
    font=dict(size=11),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(orientation='v', x=0.75, y=0.35,
                bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCC', borderwidth=1),
)

fig8.update_xaxes(range=[1945, 2026], row=2, col=1, title_text='Publication year')
fig8.update_yaxes(title_text='Abstracts per year<br>(10k sample)', row=1, col=1,
                  showgrid=True, gridcolor='#EEEEEE', zeroline=False)
fig8.update_yaxes(title_text='Rejection rate (%)', row=2, col=1,
                  range=[0, 55], showgrid=True, gridcolor='#EEEEEE', zeroline=False)

fig8.show()


In [16]:
fig8.write_image(FIG / 'fig8_yearly_rejection_rate.pdf')
fig8.write_image(FIG / 'fig8_yearly_rejection_rate.png', scale=3)
print('Saved fig8')


Saved fig8
